- _exponent._bronze_allscripts_tw_works.dbo_person_address
- _exponent._bronze_allscripts_tw_works.dbo_pharmacy_de
- _exponent._bronze_allscripts_tw_works.dbo_site_de

In [0]:
source = 'allscripts_tw'

In [0]:

dbo_person_address_df = spark.sql(f'''SELECT 
LOWER(TRIM(dbo_person_address.addressline1)) AS address_1,
LOWER(TRIM(dbo_person_address.addressline2)) AS address_2,
LOWER(TRIM(dbo_person_address.city)) AS city,
LOWER(TRIM(dbo_person_address.state)) AS state,
LOWER(TRIM(dbo_person_address.zipcode)) zip,
LOWER(TRIM(dbo_person_address.county)) AS county,
CONCAT_WS(CHR(31), '{source}','dbo_person_address', 'id', CAST(dbo_person_address.id AS BIGINT)) AS location_source_value,
-- CONCAT('{source}', ' | ', CAST(dbo_person_address.id AS INT)) AS location_source_value,
COALESCE(domain_source_to_concept.omop_concept_id, 0) AS country_concept_id,
LOWER(TRIM(dbo_person_address.country)) AS country_source_value,
NULL AS latitude,
NULL AS longitude,
'{source}' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_person_address
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept
ON LOWER(TRIM(dbo_person_address.country)) = LOWER(TRIM(domain_source_to_concept.source_value))
AND domain_source_to_concept.source_system = 'allscripts_tw' 
AND domain_source_to_concept.source_table = 'dbo_person_address' 
AND domain_source_to_concept.source_field = 'country' 
AND domain_source_to_concept.domain_id = 'Geography'
WHERE 1=1
AND dbo_person_address.addressline1 IS NOT NULL
''')
silver_df = dbo_person_address_df
silver_df.createOrReplaceTempView("silver")


In [0]:
%sql
MERGE INTO _exponent.omop_silver.location AS tgt
USING silver AS src
ON tgt.location_source_value = src.location_source_value

WHEN MATCHED AND NOT (
     tgt.address_1          <=> src.address_1
 AND tgt.address_2          <=> src.address_2
 AND tgt.city               <=> src.city
 AND tgt.state              <=> src.state
 AND tgt.zip                <=> src.zip
 AND tgt.county             <=> src.county
 AND tgt.country_concept_id <=> src.country_concept_id
 AND tgt.country_source_value            <=> src.country_source_value
 AND tgt.latitude           <=> src.latitude
 AND tgt.longitude          <=> src.longitude
 AND tgt.source_system      <=> src.source_system
) THEN UPDATE SET
  tgt.address_1          = src.address_1,
  tgt.address_2          = src.address_2,
  tgt.city               = src.city,
  tgt.state              = src.state,
  tgt.zip                = src.zip,
  tgt.county             = src.county,
  tgt.country_concept_id = src.country_concept_id,
  tgt.country_source_value            = src.country_source_value,
  tgt.latitude           = src.latitude,
  tgt.longitude          = src.longitude,
  tgt.source_system      = src.source_system,
  tgt.last_mod_tsp       = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  address_1,
  address_2,
  city,
  state,
  zip,
  county,
  location_source_value,
  country_concept_id,
  country_source_value,
  latitude,
  longitude,
  source_system,
  last_mod_tsp
) VALUES (
  src.address_1,
  src.address_2,
  src.city,
  src.state,
  src.zip,
  src.county,
  src.location_source_value,
  src.country_concept_id,
  src.country_source_value,
  src.latitude,
  src.longitude,
  src.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_location (
    source_system,
    location_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.location_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        location_source_value,
        last_mod_tsp
    FROM _exponent.omop_silver.location
    WHERE location_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_location x
  ON s.location_source_value = x.location_source_value;


In [0]:
gold_df = spark.sql("""
SELECT
  x.location_id,
  s.address_1,
  s.address_2,
  s.city,
  s.state,
  s.zip,
  s.county,
  s.country_concept_id,
  s.latitude,
  s.longitude,
  s.location_source_value,
  s.last_mod_tsp
FROM _exponent.omop_silver.location s
JOIN _exponent.omop_mapping.source_to_location x
  ON s.location_source_value = x.location_source_value
 AND x.active_flag = true
 """)
gold_df.createOrReplaceTempView("gold")


In [0]:
%sql
    
MERGE INTO _exponent.omop.location AS tgt
USING gold AS src
ON tgt.location_id = src.location_id

WHEN MATCHED AND NOT (
     tgt.address_1          <=> src.address_1
 AND tgt.address_2          <=> src.address_2
 AND tgt.city               <=> src.city
 AND tgt.state              <=> src.state
 AND tgt.zip                <=> src.zip
 AND tgt.county             <=> src.county
 AND tgt.country_concept_id <=> src.country_concept_id
 AND tgt.latitude           <=> src.latitude
 AND tgt.longitude          <=> src.longitude
 AND tgt.location_source_value <=> src.location_source_value
) THEN UPDATE SET
  tgt.address_1            = src.address_1,
  tgt.address_2            = src.address_2,
  tgt.city                 = src.city,
  tgt.state                = src.state,
  tgt.zip                  = src.zip,
  tgt.county               = src.county,
  tgt.country_concept_id   = src.country_concept_id,
  tgt.latitude             = src.latitude,
  tgt.longitude            = src.longitude,
  tgt.location_source_value = src.location_source_value

WHEN NOT MATCHED THEN INSERT (
  location_id,
  address_1,
  address_2,
  city,
  state,
  zip,
  county,
  location_source_value,
  country_concept_id,
  latitude,
  longitude
) VALUES (
  src.location_id,
  src.address_1,
  src.address_2,
  src.city,
  src.state,
  src.zip,
  src.county,
  src.location_source_value,
  src.country_concept_id,
  src.latitude,
  src.longitude
);